In [ ]:
import pandas as pd
import requests
import time

In [ ]:
file_path = "your_file_path.csv"
data = pd.read_csv(file_path)

# make backup copy
data_raw = data.copy()

# addrs Dataframe
create a new dataframe with one unique row per address

In [ ]:
addresses = data[['matched_address']].drop_duplicates().reset_index(drop=True)

for each town, state 
    --> returns a pair of coordinates if a match is found
    --> returns None, None if no match is found

In [ ]:
def get_lat_lon_from_census(address):
    base_url = "https://geocoding.geo.census.gov/geocoder/locations/onelineaddress"
    params = {
        'address': address,
        'benchmark': 'Public_AR_Current',
        'format': 'json'
    }

    response = requests.get(base_url, params=params)
    if response.status_code == 200:
        result = response.json()
        try:
            coords = result['result']['addressMatches'][0]['coordinates']
            return coords['y'], coords['x']  # (latitude, longitude)
        except (IndexError, KeyError):
            return None, None
    else:
        return None, None


matches each address with a latitude, longitude pair and adds each of these values to their respective latitude, longitude columns in the unique addresses dataframe

In [ ]:
# Apply with delay to respect usage limits
latitudes = []
longitudes = []

for _, row in addresses.iterrows():
    lat, lon = get_lat_lon_from_census(row['matched_address'])
    latitudes.append(lat)
    longitudes.append(lon)
    time.sleep(1)  # 1 second pause between requests (1 request/second limit)

addresses['latitude'] = latitudes
addresses['longitude'] = longitudes

# Merge
merge the matched addresses and coords back into the original dataframe

In [ ]:
data = data.merge(
    addresses,
    on='matched_address',
    how='left'
)

In [ ]:
data['latitude']

# Save to file

In [ ]:
data.to_csv('your_file_path.csv', index=False)